# Weighted Metrics Load Balancer Parameters

This README explains the parameters used in the `weighted_metrics_load_balancer` policy for load balancing across AI inference instances. The policy selects the best instance based on weighted metrics like queue length, active sessions, and hardware utilization.

## Core Selection Parameters

### `selection_mode`
Determines how instances are selected.
- **Options**: `"weighted"`, `"equal"`
- **Default**: `"weighted"`
- **Example**: 
  - `"weighted"`: Chooses instance with best overall score based on weights.
  - `"equal"`: Chooses instance with shortest queue length for even distribution.

### `weights`
Weights for different metrics (must sum to 1.0).
- **Default**: `{"active_sessions": 0.5, "queue_length": 0.5, "latency": 0.0, "requested_tokens_per_second": 0.0}`
- **Example**: `{"active_sessions": 0.3, "queue_length": 0.4, "latency": 0.3}` - Prioritizes queue length and latency over active sessions.

### `averaging_period`
Time period for averaging metrics.
- **Default**: `"average_1m"`
- **Example**: `"average_5m"` - Uses 5-minute averages instead of 1-minute.

### `tie_breaker`
How to break ties when multiple instances have similar scores for `weighted`.
- **Options**: `"first"`, `"round_robin"`
- **Default**: `"first"`
- **Example**: `"round_robin"` - Cycles through tied instances for fair distribution.

## Metric Configuration

### `metric_configs`
Configuration for each metric including thresholds and scoring direction.
- **Default**: Complex dict with max_threshold and invert_score for each metric.
- **Example**:
  ```json
  {
    "queue_length": {"max_threshold": 500, "invert_score": true},
    "active_sessions": {"max_threshold": 50, "invert_score": true}
  }
  ```
  - `invert_score: true` means lower values are better (higher score).

## Session Management

### `session_timeout_seconds`
How long to keep sessions cached before cleanup.
- **Default**: `3600` (1 hour)
- **Example**: `1800` - Clean up sessions after 30 minutes of inactivity.


## Redistribution Parameters

### `task_imbalance_threshold`
Threshold for considering sessions imbalanced across instances. Will be used when Block metrics are not received. This is fall back.
- **Default**: `100`
- **Example**: `50` - Redistribute if one instance has 50+ more tasks than others.


### `redistribution_percentage`
Fraction of sessions to redistribute when instances change for overloaded instances.
- **Default**: `0.2` (20%)
- **Example**: `0.5` - Move 50% of sessions when new instances are added.

### `redistribution_interval`
How often to check for redistribution (in seconds).
- **Default**: `300` (5 minutes)
- **Example**: `600` - Check every 10 minutes.

### `redistribution_selection_mode`
How to select instances for redistribution.
- **Options**: `"least_loaded"`, `"random"`, `"best"`
- **Default**: `"least_loaded"`
- **Example**: `"random"` - Randomly assign sessions during redistribution.

### `imbalance_threshold`
Minimum difference in queue lengths is needed before redistribution.If this condition is not met, no redistribution will occur. This is to avoid small fluctuations causing unnecessary redistributions.
- **Default**: `10`
- **Example**: `5` - Redistribute if any instance has 5+ fewer/more sessions than average.

### `overload_threshold_percentage`
Percentage above average load to consider an instance overloaded.
- **Default**: `0.2` (20%)
- **Example**: `0.5` - Consider overloaded if 50% above average.

## Performance Optimization

### `metrics_interval`
How often to fetch fresh metrics (in seconds).
- **Default**: `60` (1 minute)
- **Example**: `30` - Fetch metrics every 30 seconds for more responsive balancing.


## Example Configuration

```json
{
  "selection_mode": "weighted",
  "weights": {
    "queue_length": 0.6,
    "active_sessions": 0.4
  },
  "session_timeout_seconds": 1800,
  "metrics_interval": 30,
  "redistribution_percentage": 0.3,
  "imbalance_threshold": 20
}
```

This configuration prioritizes queue length, caches sessions for 30 minutes, fetches metrics every 30 seconds, and redistributes 30% of sessions if imbalance exceeds 20.